# LangChain + OpenMetadata Template
### Built by Baibhav Prateek | OpenMetadata Hackathon 2026

## What is this?
A reusable template that connects AI to OpenMetadata.
Anyone can use this as a starting point for their own
AI-powered data catalog applications.

## How to use this template:
1) Add your API keys
2) Run all cells in order
3) Ask your own questions
4) Customize the questions for your use case

## Technologies used:
1) OpenMetadata API for metadata
2) Groq AI (LLaMA 3) for natural language processing
3) Python requests for API calls

In [ ]:
import requests
import json
from groq import Groq

# Your credentials
GROQ_API_KEY = "your_groq_api_key_here"
BASE_URL = "https://sandbox.open-metadata.org"
TOKEN = "your_openmetadata_token_here"

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

print("✅ Setup complete!")

In [ ]:
# Functions to fetch data from OpenMetadata
def get_tables(limit=10):
    response = requests.get(
        f"{BASE_URL}/api/v1/tables",
        headers=HEADERS,
        params={"limit": limit}
    )
    return response.json().get("data", [])

def get_databases():
    response = requests.get(
        f"{BASE_URL}/api/v1/databases",
        headers=HEADERS,
        params={"limit": 20}
    )
    return response.json().get("data", [])

def search_assets(query):
    response = requests.get(
        f"{BASE_URL}/api/v1/search/query",
        headers=HEADERS,
        params={"q": query, "index": "table_search_index", "limit": 5}
    )
    return response.json().get("hits", {}).get("hits", [])

def get_table_details(table_name):
    results = search_assets(table_name)
    if results:
        return results[0].get("_source", {})
    return {}

print("✅ Helper functions ready!")

In [ ]:
# This function connects AI with OpenMetadata
# Step 1; First I fetch real tables from OpenMetadata
# Step 2 ; I give that information to the AI as context
# Step 3; The AI uses that context to answer the question
# This way the AI always has uptodate information

def ask_ai(question):
    # Fetch context from OpenMetadata
    tables = get_tables(limit=10)
    table_names = [t.get("name", "") for t in tables]
    
    # Build prompt
    prompt = f"""You are a helpful data catalog assistant.
You have access to OpenMetadata with these tables: {table_names}

User question: {question}

Answer helpfully and concisely."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test it!
answer = ask_ai("How many tables do we have and what are some of their names?")
print("🤖 AI says:")
print(answer)

In [ ]:
# Interactive Q&A session
questions = [
    "Which tables look incomplete or poorly documented?",
    "What kind of organization does this data belong to?",
    "If you were a new data analyst, which tables would you explore first?",
]

print("=" * 60)
print("   🤖 OpenMetadata AI Assistant Demo")
print("=" * 60)

for question in questions:
    print(f"\n❓ Question: {question}")
    print("-" * 40)
    answer = ask_ai(question)
    print(f"🤖 Answer: {answer}")
    print()

print("=" * 60)
print("   🤖 OpenMetadata AI Template Demo")
print("   Built for OpenMetadata Hackathon 2026")
print("=" * 60)